In [1]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [2]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aaronrao/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
df = pd.read_csv("D:\vs codes\Daily Challenge\60 days daily challenge\day41\IMDB Dataset.csv")

print("Dataset Preview:")
df.head()

Dataset Preview:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
print("Shape:", df.shape)
print("\nColumns:", df.columns)
print("\nClass Distribution:")
print(df["sentiment"].value_counts())

Shape: (50000, 2)

Columns: Index(['review', 'sentiment'], dtype='object')

Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [6]:
def clean_text(text):
    text = str(text).lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

In [7]:
df["processed_text"] = df["review"].apply(clean_text)

df[["review", "processed_text"]].head()

,review,processed_text
0,One of the other reviewers has mentioned that ...,one reviewers mentioned watching 1 oz episode ...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically theres family little boy jake thinks...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


In [8]:
X = df["processed_text"]
y = df["sentiment"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 40000
Test size: 10000


In [10]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)   
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF Shape:", X_train_tfidf.shape)

TF-IDF Shape: (40000, 10000)


In [11]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

print("Model trained successfully")

/Users/aaronrao/Desktop/projects/Global Coding Challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/aaronrao/Desktop/projects/Global Coding Challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/aaronrao/Desktop/projects/Global Coding Challenge/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


Model trained successfully


In [12]:
y_pred = model.predict(X_test_tfidf)

print("Sample Predictions:")
print(y_pred[:10])

Sample Predictions:
['negative' 'positive' 'negative' 'positive' 'negative' 'positive'
 'positive' 'negative' 'negative' 'negative']


In [13]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8944

Classification Report:
              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      4961
    positive       0.89      0.91      0.90      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [14]:
sample_reviews = [
    "This movie was absolutely amazing",
    "Worst film I have ever seen",
    "It was okay, not great"
]
sample_clean = [clean_text(text) for text in sample_reviews]
sample_tfidf = vectorizer.transform(sample_clean)
predictions = model.predict(sample_tfidf)
print("\nCustom Predictions:")
for text, pred in zip(sample_reviews, predictions):
    print(f"{text} → {pred}")


Custom Predictions:
This movie was absolutely amazing → positive
Worst film I have ever seen → negative
It was okay, not great → positive


In [15]:
import joblib
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
joblib.dump(model, "sentiment_model.pkl")
print("Model and vectorizer saved successfully")

Model and vectorizer saved successfully
